# Klasifikasi Teks dengan TensorFlow/Keras

Notebook ini adalah versi berbahasa Indonesia yang disederhanakan dan diperluas dari tutorial **Basic text classification** TensorFlow. Tutorial aslinya dapat diakses [di sini](https://www.tensorflow.org/tutorials/keras/text_classification).

Kita akan membuat model untuk membaca review film dan memprediksi apakah review tersebut bernada:

- **negatif**
- **positif**

Jenis masalah ini disebut **binary text classification**, karena kelas target hanya ada dua.

---

## Gambaran Besar Proses

Secara konseptual, prosesnya seperti ini:

```text
Review teks mentah
        ↓
Pembersihan teks
        ↓
Tokenisasi dan pengubahan kata menjadi angka
        ↓
Embedding
        ↓
Model neural network
        ↓
Probabilitas review positif
        ↓
Prediksi: negatif / positif
```

Komputer tidak memahami kata secara langsung. Karena itu, teks harus diubah menjadi representasi numerik terlebih dahulu.

## 1. Persiapan Library

Kita akan menggunakan:

- `tensorflow` untuk membangun dan melatih model.
- `keras.layers` untuk membuat layer neural network.
- `matplotlib` untuk membuat grafik.
- `os`, `shutil`, `re`, dan `string` untuk mengelola file dan membersihkan teks.

> Catatan: Jika TensorFlow belum terpasang, jalankan perintah berikut di terminal atau command prompt:
>
> ```bash
> pip install tensorflow matplotlib
> ```

In [ ]:
import os
import re
import shutil
import string

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import losses

print("TensorFlow version:", tf.__version__)

## 2. Mengunduh Dataset IMDB

Kita menggunakan **Large Movie Review Dataset** dari Stanford.

Dataset ini berisi review film dari IMDB yang sudah diberi label:

- `pos`: review positif
- `neg`: review negatif

Secara umum, dataset ini berisi 50.000 review:
- 25.000 review untuk training
- 25.000 review untuk testing

TensorFlow akan mengunduh file dataset secara otomatis saat sel berikut dijalankan.

In [ ]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

dataset_path = tf.keras.utils.get_file(
    fname="aclImdb_v1",
    origin=url,
    untar=True,
    cache_dir=".",
    cache_subdir=""
)

dataset_dir = os.path.join(os.path.dirname(dataset_path), "aclImdb")

print("Lokasi dataset:", dataset_dir)
print("Isi folder dataset:", os.listdir(dataset_dir))

## 3. Mengecek Struktur Folder Dataset

Untuk menggunakan `text_dataset_from_directory`, struktur folder harus berbentuk seperti ini:

```text
folder_utama/
    kelas_1/
        file_teks_1.txt
        file_teks_2.txt
    kelas_2/
        file_teks_3.txt
        file_teks_4.txt
```

Pada dataset IMDB, folder yang kita butuhkan adalah:

```text
aclImdb/train/pos
aclImdb/train/neg
aclImdb/test/pos
aclImdb/test/neg
```

Namun, di dalam `aclImdb/train` terdapat folder tambahan bernama `unsup` yang tidak memiliki label untuk klasifikasi biner. Karena itu, folder tersebut perlu dihapus agar proses pembacaan dataset menjadi bersih.

In [ ]:
train_dir = os.path.join(dataset_dir, "train")
test_dir = os.path.join(dataset_dir, "test")

print("Isi folder train sebelum pembersihan:")
print(os.listdir(train_dir))

unsup_dir = os.path.join(train_dir, "unsup")

# Hapus folder unsup jika masih ada.
# Kode ini aman dijalankan berulang kali.
if os.path.isdir(unsup_dir):
    shutil.rmtree(unsup_dir)
    print("Folder 'unsup' berhasil dihapus.")
else:
    print("Folder 'unsup' sudah tidak ada.")

print("\nIsi folder train setelah pembersihan:")
print(os.listdir(train_dir))

## 4. Melihat Contoh Isi File Review

Setiap review disimpan sebagai satu file `.txt`.

Mari kita buka salah satu review positif agar mahasiswa memahami bentuk data mentahnya.

In [ ]:
sample_file = os.path.join(train_dir, "pos", "1181_9.txt")

with open(sample_file, encoding="utf-8") as f:
    sample_text = f.read()

print(sample_text[:1000])

## 5. Membuat Dataset Training, Validation, dan Testing

Dalam machine learning, dataset biasanya dibagi menjadi tiga bagian:

| Bagian Data | Fungsi |
|---|---|
| Training data | Dipakai model untuk belajar |
| Validation data | Dipakai untuk memantau performa selama training |
| Test data | Dipakai untuk evaluasi akhir setelah training selesai |

Dataset IMDB sudah memiliki folder `train` dan `test`, tetapi belum memiliki folder validation. Karena itu, kita akan mengambil 20% dari data training sebagai validation set.

In [ ]:
batch_size = 32
seed = 42

raw_train_ds = tf.keras.utils.text_dataset_from_directory(
    directory=train_dir,
    batch_size=batch_size,
    validation_split=0.2,
    subset="training",
    seed=seed
)

raw_val_ds = tf.keras.utils.text_dataset_from_directory(
    directory=train_dir,
    batch_size=batch_size,
    validation_split=0.2,
    subset="validation",
    seed=seed
)

raw_test_ds = tf.keras.utils.text_dataset_from_directory(
    directory=test_dir,
    batch_size=batch_size
)

print("\nNama kelas:", raw_train_ds.class_names)

## 6. Memahami Label Dataset

Biasanya `text_dataset_from_directory` memberi label berdasarkan urutan nama folder secara alfabetis.

Pada dataset ini biasanya:

```text
0 = neg
1 = pos
```

Mari kita tampilkan beberapa contoh review dan labelnya.

In [ ]:
for text_batch, label_batch in raw_train_ds.take(1):
    for i in range(3):
        print("Review:")
        print(text_batch.numpy()[i][:500])
        print("Label angka:", label_batch.numpy()[i])
        print("Label kelas:", raw_train_ds.class_names[label_batch.numpy()[i]])
        print("-" * 80)

## 7. Pembersihan Teks

Teks mentah biasanya memiliki beberapa hal yang kurang ideal, misalnya:

- huruf besar dan kecil yang tidak konsisten,
- tag HTML seperti `<br />`,
- tanda baca yang mungkin tidak kita perlukan untuk model sederhana.

Kita akan membuat fungsi `custom_standardization` untuk:

1. mengubah teks menjadi huruf kecil,
2. mengganti `<br />` dengan spasi,
3. menghapus tanda baca.

> Penting: Ini bukan satu-satunya cara preprocessing teks. Untuk kasus nyata, preprocessing harus disesuaikan dengan bahasa dan domain data.

In [ ]:
@tf.keras.utils.register_keras_serializable(package="Custom")
def custom_standardization(input_text):
    # 1. Ubah semua huruf menjadi huruf kecil
    lowercase = tf.strings.lower(input_text)

    # 2. Ganti tag HTML <br /> dengan spasi
    no_html = tf.strings.regex_replace(lowercase, "<br />", " ")

    # 3. Hapus tanda baca
    return tf.strings.regex_replace(
        no_html,
        "[%s]" % re.escape(string.punctuation),
        ""
    )

In [ ]:
contoh_teks = tf.constant(["This movie was GREAT!!!<br />I really loved it."])

print("Sebelum pembersihan:")
print(contoh_teks.numpy()[0].decode("utf-8"))

print("\nSetelah pembersihan:")
print(custom_standardization(contoh_teks).numpy()[0].decode("utf-8"))

## 8. TextVectorization: Mengubah Teks Menjadi Angka

Model neural network tidak bisa langsung menerima kalimat seperti:

```text
this movie was great
```

Kita harus mengubahnya menjadi angka, misalnya:

```text
[12, 45, 8, 91]
```

Layer `TextVectorization` melakukan beberapa proses penting:

1. **Standardization**: membersihkan teks.
2. **Tokenization**: memecah teks menjadi token/kata.
3. **Indexing**: memberi nomor unik untuk setiap kata.
4. **Padding/Truncating**: menyamakan panjang input.

Parameter penting:

- `max_tokens`: jumlah maksimum kata yang disimpan dalam kosakata.
- `output_mode='int'`: hasilnya berupa urutan angka.
- `output_sequence_length`: panjang setiap review setelah dipotong atau diisi padding.

Contoh:
- Review pendek akan ditambah padding.
- Review terlalu panjang akan dipotong.

In [ ]:
max_features = 10000      # Maksimum jumlah kata dalam vocabulary
sequence_length = 250     # Panjang maksimum setiap review setelah vectorization

vectorize_layer = layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=max_features,
    output_mode="int",
    output_sequence_length=sequence_length
)

# Ambil hanya teks dari dataset training, tanpa label.
train_text = raw_train_ds.map(lambda text, label: text)

# adapt() membuat vocabulary berdasarkan data training.
# Jangan gunakan validation/test data untuk adapt, karena itu menyebabkan data leakage.
vectorize_layer.adapt(train_text)

print("Jumlah kata dalam vocabulary:", len(vectorize_layer.get_vocabulary()))
print("10 kata pertama dalam vocabulary:")
print(vectorize_layer.get_vocabulary()[:10])

## 9. Melihat Hasil Vectorization

Sekarang kita lihat bagaimana review mentah berubah menjadi deretan angka.

Perhatikan bahwa setiap kata akan diganti dengan indeks dari vocabulary.

In [ ]:
for text_batch, label_batch in raw_train_ds.take(1):
    first_review = text_batch[0]
    first_label = label_batch[0]

    print("Review asli:")
    print(first_review.numpy()[:500])

    print("\nLabel:", raw_train_ds.class_names[first_label.numpy()])

    vectorized_review = vectorize_layer(tf.expand_dims(first_review, -1))

    print("\n20 token pertama setelah vectorization:")
    print(vectorized_review.numpy()[0][:20])
    break

In [ ]:
vocab = vectorize_layer.get_vocabulary()

print("Contoh mapping indeks ke kata:")
for index in [1, 2, 3, 10, 100, 1000]:
    if index < len(vocab):
        print(f"{index} -> {vocab[index]}")

## 10. Menyiapkan Dataset yang Sudah Berupa Angka

Sekarang kita ubah semua dataset:

- training
- validation
- testing

dari bentuk teks mentah menjadi angka.

Setelah proses ini, dataset siap masuk ke model neural network.

In [ ]:
def vectorize_text(text, label):
    text = tf.expand_dims(text, -1)
    return vectorize_layer(text), label

train_ds = raw_train_ds.map(vectorize_text)
val_ds = raw_val_ds.map(vectorize_text)
test_ds = raw_test_ds.map(vectorize_text)

for x_batch, y_batch in train_ds.take(1):
    print("Shape input batch:", x_batch.shape)
    print("Shape label batch:", y_batch.shape)
    print("Contoh input pertama, 20 angka pertama:")
    print(x_batch.numpy()[0][:20])

## 11. Optimasi Pipeline Data

Dua fungsi berikut sering dipakai agar training lebih efisien:

- `cache()`: menyimpan data yang sudah diproses agar tidak dihitung ulang terus-menerus.
- `prefetch()`: menyiapkan batch berikutnya saat model sedang training batch saat ini.

Ini berguna agar model tidak terlalu sering menunggu proses input data.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 12. Membuat Model Neural Network Sederhana

Model yang digunakan terdiri dari beberapa layer:

```text
Embedding
    ↓
Dropout
    ↓
GlobalAveragePooling1D
    ↓
Dropout
    ↓
Dense sigmoid
```

Penjelasan singkat:

| Layer | Fungsi |
|---|---|
| `Embedding` | Mengubah indeks kata menjadi vektor dense |
| `Dropout` | Mengurangi risiko overfitting |
| `GlobalAveragePooling1D` | Merangkum semua vektor kata menjadi satu vektor review |
| `Dense(1, sigmoid)` | Menghasilkan probabilitas review positif |

Output model adalah angka antara 0 dan 1.

Interpretasi sederhana:

```text
mendekati 0 → negatif
mendekati 1 → positif
```

In [ ]:
embedding_dim = 16

model = tf.keras.Sequential([
    layers.Embedding(input_dim=max_features, output_dim=embedding_dim),
    layers.Dropout(0.2),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid")
])

model.summary()

## 13. Compile Model

Sebelum training, model perlu dikonfigurasi dengan:

- **loss function**: fungsi untuk mengukur seberapa salah prediksi model.
- **optimizer**: algoritma untuk memperbaiki bobot model.
- **metric**: ukuran performa yang mudah dibaca manusia.

Karena ini adalah klasifikasi biner, kita gunakan:

```python
BinaryCrossentropy
```

dan metrik:

```python
BinaryAccuracy
```

In [ ]:
model.compile(
    loss=losses.BinaryCrossentropy(),
    optimizer="adam",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5)]
)

## 14. Training Model

Pada tahap ini, model belajar dari review film.

Untuk pemula, kita gunakan jumlah epoch yang tidak terlalu besar agar proses training tidak terlalu lama.

> Jika menggunakan CPU, proses ini mungkin memerlukan beberapa menit.

In [ ]:
epochs = 5

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

## 15. Evaluasi Model pada Test Set

Setelah training selesai, kita evaluasi model menggunakan test set.

Test set adalah data yang tidak dipakai saat training maupun validation, sehingga lebih adil untuk mengukur kemampuan generalisasi model.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

## 16. Visualisasi Loss dan Akurasi

Grafik membantu kita memahami proses training.

Hal yang perlu diperhatikan:

- Jika training accuracy naik tetapi validation accuracy berhenti naik, model mungkin mulai overfitting.
- Jika training loss turun tetapi validation loss naik, itu juga tanda overfitting.

In [ ]:
history_dict = history.history

train_loss = history_dict["loss"]
val_loss = history_dict["val_loss"]
train_acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]

epochs_range = range(1, len(train_loss) + 1)

plt.figure()
plt.plot(epochs_range, train_loss, label="Training loss")
plt.plot(epochs_range, val_loss, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training dan Validation Loss")
plt.legend()
plt.show()

In [ ]:
plt.figure()
plt.plot(epochs_range, train_acc, label="Training accuracy")
plt.plot(epochs_range, val_acc, label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training dan Validation Accuracy")
plt.legend()
plt.show()

## 17. Confusion Matrix

Akurasi memberikan gambaran umum, tetapi kita juga perlu tahu jenis kesalahan model.

Untuk klasifikasi biner, confusion matrix biasanya berbentuk:

| | Prediksi Negatif | Prediksi Positif |
|---|---:|---:|
| Aktual Negatif | True Negative | False Positive |
| Aktual Positif | False Negative | True Positive |

Interpretasi:
- **True Negative**: review negatif diprediksi negatif.
- **False Positive**: review negatif salah diprediksi positif.
- **False Negative**: review positif salah diprediksi negatif.
- **True Positive**: review positif diprediksi positif.

In [ ]:
y_true = []
y_pred = []

for x_batch, y_batch in test_ds:
    probabilities = model.predict(x_batch, verbose=0).reshape(-1)
    predictions = (probabilities >= 0.5).astype("int32")

    y_true.extend(y_batch.numpy())
    y_pred.extend(predictions)

cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=2).numpy()

print("Confusion Matrix:")
print(cm)

plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks([0, 1], ["neg", "pos"])
plt.yticks([0, 1], ["neg", "pos"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.show()

## 18. Membuat Model End-to-End untuk Prediksi Teks Mentah

Sampai saat ini, model utama menerima input berupa angka hasil `TextVectorization`.

Agar lebih praktis, kita buat model baru yang menerima teks mentah langsung.

Strukturnya:

```text
Teks mentah
    ↓
TextVectorization
    ↓
Model klasifikasi
    ↓
Probabilitas review positif
```

Ini berguna ketika model akan digunakan di aplikasi nyata.

In [ ]:
export_model = tf.keras.Sequential([
    vectorize_layer,
    model
])

export_model.compile(
    loss=losses.BinaryCrossentropy(),
    optimizer="adam",
    metrics=["accuracy"]
)

metrics = export_model.evaluate(raw_test_ds, return_dict=True)
print(metrics)

## 19. Prediksi Review Baru

Sekarang kita coba masukkan kalimat review buatan sendiri.

Aturan interpretasi:

```text
probabilitas >= 0.5 → positif
probabilitas < 0.5  → negatif
```

In [ ]:
def predict_review(text):
    probability = float(export_model.predict(tf.constant([text]), verbose=0)[0][0])
    label = "positif" if probability >= 0.5 else "negatif"

    print("Review:")
    print(text)
    print(f"Probabilitas positif: {probability:.4f}")
    print("Prediksi:", label)
    print("-" * 80)

examples = [
    "The movie was fantastic. I loved the story and the acting.",
    "The film was boring, too long, and not interesting at all.",
    "It was okay. Some parts were good, but some parts were disappointing.",
    "A wonderful movie with strong characters and beautiful music.",
    "I regret watching this movie. The plot was terrible."
]

for example in examples:
    predict_review(example)

## 20. Menyimpan Model

Model end-to-end dapat disimpan agar bisa digunakan kembali tanpa training ulang.

Format `.keras` adalah format penyimpanan model Keras modern.

> Catatan: Karena kita memakai fungsi preprocessing kustom, fungsi tersebut sudah didaftarkan menggunakan `@tf.keras.utils.register_keras_serializable`.

In [ ]:
export_model.save("model_sentimen_imdb.keras")
print("Model berhasil disimpan sebagai model_sentimen_imdb.keras")

## 21. Memuat Kembali Model

Sel berikut bersifat opsional. Gunakan untuk memastikan model yang disimpan dapat dibuka kembali.

In [ ]:
loaded_model = tf.keras.models.load_model("model_sentimen_imdb.keras")

predict_text = tf.constant(["This movie was amazing and very enjoyable."])
loaded_prediction = loaded_model.predict(predict_text, verbose=0)

print("Prediksi dari model yang dimuat ulang:", loaded_prediction[0][0])

# Latihan

## Latihan 1. Prediksi Kalimat Sendiri

Ubah isi variabel `my_review` dengan review buatan Anda sendiri. Jalankan prediksi dan jelaskan hasilnya.

Pertanyaan refleksi:

1. Apakah prediksi model sesuai ekspektasi Anda?
2. Apakah kalimat netral lebih sulit diprediksi dibanding kalimat sangat positif atau sangat negatif?

In [ ]:
my_review = "This movie was not perfect, but I enjoyed many parts of it."

predict_review(my_review)

## Latihan 2. Ubah `embedding_dim`

Pada model awal, kita menggunakan:

```python
embedding_dim = 16
```

Cobalah nilai lain:

```python
embedding_dim = 8
embedding_dim = 32
embedding_dim = 64
```

Catat:

1. Apakah training menjadi lebih lambat?
2. Apakah validation accuracy meningkat?
3. Apakah model mulai overfitting?

## Latihan 3. Ubah Panjang Sequence

Pada vectorization, kita menggunakan:

```python
sequence_length = 250
```

Cobalah nilai:

```python
sequence_length = 100
sequence_length = 500
```

Diskusikan:

1. Apa yang terjadi jika sequence terlalu pendek?
2. Apa konsekuensi jika sequence terlalu panjang?
3. Apakah akurasi berubah signifikan?

## Latihan 4. Gunakan EarlyStopping

Tambahkan callback berikut saat `model.fit()`:

```python
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)
```

Lalu panggil:

```python
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stopping]
)
```

Tugas:

1. Amati pada epoch ke berapa training berhenti.
2. Jelaskan mengapa EarlyStopping dapat membantu mengurangi overfitting.

## Latihan 5. Ubah Threshold Klasifikasi

Saat ini threshold yang digunakan adalah:

```text
0.5
```

Artinya:
- probabilitas positif ≥ 0.5 → positif
- probabilitas positif < 0.5 → negatif

Cobalah threshold:

```python
0.4
0.6
0.7
```

Pertanyaan:

1. Apakah jumlah prediksi positif berubah?
2. Bagaimana dampaknya terhadap confusion matrix?
3. Dalam aplikasi nyata, kapan kita perlu menaikkan atau menurunkan threshold?

In [ ]:
# Contoh eksperimen threshold

threshold = 0.6

y_true_threshold = []
y_pred_threshold = []

for x_batch, y_batch in test_ds:
    probabilities = model.predict(x_batch, verbose=0).reshape(-1)
    predictions = (probabilities >= threshold).astype("int32")

    y_true_threshold.extend(y_batch.numpy())
    y_pred_threshold.extend(predictions)

cm_threshold = tf.math.confusion_matrix(
    y_true_threshold,
    y_pred_threshold,
    num_classes=2
).numpy()

print(f"Confusion Matrix dengan threshold = {threshold}")
print(cm_threshold)

# Ringkasan

Pada notebook ini, kita telah mempelajari pipeline dasar klasifikasi teks menggunakan TensorFlow/Keras:

```text
Download dataset
    ↓
Baca file teks dari folder
    ↓
Bagi data menjadi train/validation/test
    ↓
Bersihkan teks
    ↓
Ubah teks menjadi angka dengan TextVectorization
    ↓
Bangun model Embedding sederhana
    ↓
Training model
    ↓
Evaluasi model
    ↓
Prediksi teks baru
```

Konsep paling penting:

1. **Teks harus diubah menjadi angka** sebelum masuk ke model.
2. **TextVectorization** membuat vocabulary dan mengubah kata menjadi indeks.
3. **Embedding** mengubah indeks kata menjadi vektor yang dipelajari selama training.
4. **Validation set** membantu memantau generalisasi model.
5. **Overfitting** terjadi ketika model terlalu hafal data training tetapi kurang baik pada data baru.
6. **Model end-to-end** memudahkan prediksi karena bisa menerima teks mentah secara langsung.

# Referensi

- TensorFlow — Basic text classification: https://www.tensorflow.org/tutorials/keras/text_classification
- TensorFlow API — `text_dataset_from_directory`: https://www.tensorflow.org/api_docs/python/tf/keras/utils/text_dataset_from_directory
- TensorFlow API — `TextVectorization`: https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization
- TensorFlow API — `Embedding`: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding